# Interactive Kinetic Theory — Notebook 01
## The Drifting Maxwellian

**Course:** Kinetic Theory of Plasmas
**Format:** problem statement + interactive numerical exploration

This notebook accompanies **Problem 1 (Drifting Maxwellian)**. The derivations are left to
you (the student); here you can *play* with the distribution functions, choose physical
parameters (species, temperature, density, drift velocity) and see how the quantities
introduced in the problem behave.

### Learning objectives
After working through this notebook you should be able to:

1. distinguish the isotropic Maxwellian $f_0$, the anisotropic correction $\mathbf f_1$,
   and the two-term (drifting) expansion $f \simeq f_0 + \mathbf f_1\cdot\hat{\mathbf v}$;
2. explain why the expansion requires $|\mathbf V| \ll v_{th}$, with $v_{th}=\sqrt{kT/m}$;
3. read the speed distribution $f_s(v)$, the energy distribution $f_e(u)$ and the energy
   probability function $f_p(u)$, and locate their maxima;
4. recognise the exponential (log-linear) character of $f_p$.


---
## The problem

Let us consider a distribution function of the form
$$f(\mathbf r,\mathbf v,t)=n\left(\frac{m}{2\pi k T}\right)^{3/2}
\exp\!\left[-\frac{m(\mathbf v-\mathbf V)^2}{2kT}\right],$$
where $n(\mathbf r,t)$, $T(\mathbf r,t)$ and $\mathbf V(\mathbf r,t)$ are the particle density,
temperature and bulk velocity, respectively.

**1.** Prove that for $|\mathbf V(\mathbf r,t)|\ll\sqrt{kT/m}$ one can approximate the
distribution function by the two-term expansion
$$f(\mathbf r,\mathbf v,t)\approx f_0(\mathbf r,v,t)+\mathbf f_1(\mathbf r,v,t)\cdot\hat{\mathbf v},$$
with
$$f_0(\mathbf r,v,t)=n\left(\frac{m}{2\pi kT}\right)^{3/2}e^{-mv^2/2kT},
\qquad
\mathbf f_1(\mathbf r,v,t)=n\,v\,\frac{m}{kT}\left(\frac{m}{2\pi kT}\right)^{3/2}
e^{-mv^2/2kT}\,\mathbf V .$$

1. Prove that the maximum of the anisotropic part is
   $$|\mathbf f_1|_{\max}=n\sqrt{\frac{m}{ekT}}\left(\frac{m}{2\pi kT}\right)^{3/2}|\mathbf V|,
   \qquad\text{reached at } v=\sqrt{\frac{kT}{m}}.$$
2. Prove that the average speed of the isotropic distribution is
   $$\langle v\rangle=\frac1n\int_{\mathbb R^3}|\mathbf v|\,f_0\,d\mathbf v
   =\left(\frac{8kT}{\pi m}\right)^{1/2}.$$
3. Prove that the *speed distribution function* can be written as
   $$f_s(\mathbf r,v,t)\,dv=4\pi v^2 n\left(\frac{m}{2\pi kT}\right)^{3/2}
   e^{-mv^2/2kT}\,dv,$$
   and show that its most probable speed is
   $v=\left(2kT/m\right)^{1/2}$.
4. With the energy $u=\tfrac m2 v^2$, prove that the *energy distribution function*
   $$f_e(\mathbf r,u,t)=\frac{2n}{\sqrt\pi}\left(\frac{1}{kT}\right)^{3/2}
   u^{1/2}e^{-u/kT},$$
   and show that its most probable energy is $u=kT/2$.
5. The *energy probability function* is defined as
   $$f_p(\mathbf r,u,t)=u^{-1/2}f_e(\mathbf r,u,t).$$
   Prove that $\ln f_p$ is linear in $u$.

> **Note.** In item 5 the statement contains a small typo: it must read
> $f_p=u^{-1/2}f_e$ (the factor multiplies $f_e$, not $f_p$).

Throughout the notebook the drift is taken along $x$, i.e. $\mathbf V = V_x\,\mathbf e_x$.


---
## 0. Setup: constants, species and helper functions

Everything is evaluated in **SI units** (velocity in m/s, $f$ in m$^{-6}$s$^3$, ...).
The temperature is entered in **eV** (converted through $kT=q_e T_{\rm eV}$) and the
density in **m$^{-3}$**.

Run the four cells below once; afterwards the sliders drive every plot in the notebook.


In [1]:
# Core scientific stack
import numpy as np
from scipy import constants as const
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Render figures inside the notebook
%matplotlib inline

# Nicer default figure style
plt.rcParams.update({
    "figure.figsize": (11, 4.2),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

In [2]:
# Fundamental constants (SI)
q_e = const.e        # elementary charge          [C]
m_e = const.m_e      # electron mass              [kg]
u   = const.u        # atomic mass unit           [kg]
kB  = const.k        # Boltzmann constant         [J/K]

# Species available in the dashboard: name -> mass [kg].
# All ions are singly charged (charge state Z = 1).
SPECIES = {
    "Electron":           m_e,
    "Hydrogen ion (H+)":  1.00784 * u,
    "Helium ion (He+)":   4.00260 * u,
    "Argon ion (Ar+)":   39.94800 * u,
    "Krypton ion (Kr+)": 83.79800 * u,
    "Xenon ion (Xe+)":  131.29300 * u,
}

for _name, _mass in SPECIES.items():
    print(f"{_name:22s}  m = {_mass:.4e} kg")

Electron                m = 9.1094e-31 kg
Hydrogen ion (H+)       m = 1.6736e-27 kg
Helium ion (He+)        m = 6.6465e-27 kg
Argon ion (Ar+)         m = 6.6335e-26 kg
Krypton ion (Kr+)       m = 1.3915e-25 kg
Xenon ion (Xe+)         m = 2.1802e-25 kg


In [3]:
def kT_from_T(T_eV):
    """Thermal energy kT [J] from a temperature T given in eV."""
    return q_e * T_eV

def v_thermal(m, kT):
    """Thermal speed v_th = sqrt(kT/m) [m/s] (location of the f1 peak)."""
    return np.sqrt(kT / m)

def v_most_probable(m, kT):
    """Most probable speed v_mp = sqrt(2 kT/m) [m/s] (peak of f_s)."""
    return np.sqrt(2.0 * kT / m)

def v_mean(m, kT):
    """Mean speed <v> = sqrt(8 kT/(pi m)) [m/s]."""
    return np.sqrt(8.0 * kT / (np.pi * m))


# ---------------------------------------------------------------------
# Distribution functions in SI units.
#
# f is the phase-space density: number of particles per unit velocity
# volume, units m^-6 s^3, so that  integral f d^3v = n  [m^-3].
# ---------------------------------------------------------------------

def f0_si(v, n, m, kT):
    """Isotropic Maxwellian f0(v), v = speed [m/s]."""
    return n * (m / (2*np.pi*kT))**1.5 * np.exp(-m * v**2 / (2*kT))

def f_drift_si(vx, n, m, kT, Vx):
    """Exact drifting Maxwellian on the v_x axis (drift along x)."""
    return n * (m / (2*np.pi*kT))**1.5 * np.exp(-m * (vx - Vx)**2 / (2*kT))

def f1_radial_si(v, n, m, kT):
    """Radial coefficient |f1|(v):  f1_vec = f1_radial(v) * V  [m^-6 s^3]."""
    return n * v * (m/kT) * (m/(2*np.pi*kT))**1.5 * np.exp(-m * v**2 / (2*kT))

def two_term_si(vx, n, m, kT, Vx):
    """Two-term approximation f0 + f1.vhat on the v_x axis."""
    v = np.abs(vx)
    return f0_si(v, n, m, kT) + f1_radial_si(v, n, m, kT) * Vx * np.sign(vx)

def speed_dist_si(v, n, m, kT):
    """Speed distribution f_s(v) = 4 pi v^2 f0(v)  [m^-4 s]."""
    return 4*np.pi * v**2 * f0_si(v, n, m, kT)

def energy_dist_si(u, n, m, kT):
    """Energy distribution f_e(u)  [m^-3 J^-1],  u = m v^2 / 2  [J]."""
    return (2*n/np.sqrt(np.pi)) * kT**-1.5 * np.sqrt(u) * np.exp(-u/kT)

def energy_prob_si(u, n, m, kT):
    """Energy probability function f_p(u) = f_e / sqrt(u)  [m^-3 J^-3/2]."""
    return (2*n/np.sqrt(np.pi)) * kT**-1.5 * np.exp(-u/kT)

def g0_si(u, n, m, kT):
    """g0(u): the Maxwellian written as a function of the energy [m^-6 s^3]."""
    return n * (m/(2*np.pi*kT))**1.5 * np.exp(-u/kT)


print("helper functions defined.")

helper functions defined.


In [6]:
# ---------------------------------------------------------------------
# Dashboard controls, shared by every interactive plot below.
# ---------------------------------------------------------------------
_style = {"description_width": "120px"}

w_species = widgets.Dropdown(
    options=list(SPECIES.keys()), value="Hydrogen ion (H+)",
    description="Species", style=_style)

w_T = widgets.FloatLogSlider(
    value=10.0, base=10, min=-1, max=3, step=0.01,
    description="T [eV]", style=_style, continuous_update=False)

w_n = widgets.FloatLogSlider(
    value=1e19, base=10, min=14, max=24, step=0.01,
    description="n [m^-3]", style=_style, continuous_update=False)

w_V = widgets.FloatSlider(
    value=0.10, min=0.0, max=1.20, step=0.01,
    description="Vx/v_th", style=_style, continuous_update=False)

# The keys must match the argument names of the plotting functions.
controls = dict(species=w_species, T_eV=w_T, n=w_n, V_frac=w_V)

dashboard = widgets.VBox([
    widgets.HTML("<b>Interactive parameters</b> (drive every plot in this notebook)"),
    widgets.HBox([w_species, w_T]),
    widgets.HBox([w_n, w_V]),
])
display(dashboard)

def plot_speed(species, T_eV, n, V_frac):
    m = SPECIES[species]
    kT = kT_from_T(T_eV)
    vth = v_thermal(m, kT)
    vmp = v_most_probable(m, kT)
    vmean = v_mean(m, kT)

    v = np.linspace(0, 4*vth, 800)
    fs = speed_dist_si(v, n, m, kT)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

    ax = axes[0]                                   # ---- SI ----
    ax.plot(v, fs, lw=2.2, color="C0", label="$f_s(v)$")
    ax.axvline(vth,   color="C1", ls="--", lw=1.3, label="$v_{th}$")
    ax.axvline(vmp,   color="C2", ls="--", lw=1.3, label="$v_{mp}$")
    ax.axvline(vmean, color="C3", ls="--", lw=1.3, label="$\\langle v\\rangle$")
    ax.set_xlabel("$v$  [m/s]")
    ax.set_ylabel("$f_s$  [m$^{-4}$ s]")
    ax.set_title("SI units")
    ax.legend(fontsize=9)

    ax = axes[1]                                   # ---- normalized ----
    ax.plot(v/vth, fs*vth/n, lw=2.2, color="C0")
    for x0, c in [(1.0, "C1"), (np.sqrt(2), "C2"), (np.sqrt(8/np.pi), "C3")]:
        ax.axvline(x0, color=c, ls="--", lw=1.3)
    ax.set_xlabel("$v/v_{th}$")
    ax.set_ylabel("$f_s\\,v_{th}/n$")
    ax.set_title("Normalized units")

    fig.suptitle(f"{species},   T = {T_eV:g} eV    "
                 f"($v_{{mp}}/v_{{th}}=\\sqrt{2}$, "
                 f"$\\langle v\\rangle/v_{{th}}=\\sqrt{{8/\\pi}}\\approx1.596$)")
    fig.tight_layout()
    plt.show()

display(widgets.interactive_output(plot_speed, controls))

In [7]:
def plot_cut(species, T_eV, n, V_frac):
    m = SPECIES[species]
    kT = kT_from_T(T_eV)
    vth = v_thermal(m, kT)
    Vx = V_frac * vth                      # drift along x

    vx = np.linspace(-4*vth, 4*vth, 800)
    exact   = f_drift_si(vx, n, m, kT, Vx)
    iso     = f0_si(np.abs(vx), n, m, kT)
    twoterm = two_term_si(vx, n, m, kT, Vx)
    f1part  = f1_radial_si(np.abs(vx), n, m, kT) * Vx * np.sign(vx)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

    ax = axes[0]                                   # ---- SI ----
    ax.plot(vx, exact,   lw=2.2, color="C0", label="exact $f$")
    ax.plot(vx, iso,     lw=1.6, color="C1", ls="--", label="$f_0$ (isotropic)")
    ax.plot(vx, twoterm, lw=1.6, color="C2", ls="-.", label="$f_0 + f_1\cdot\hat{v}$")
    ax.plot(vx, f1part,  lw=1.2, color="C3", ls=":",  label="$f_1\cdot\hat{v}$ alone")
    ax.axvline(Vx, color="grey", lw=0.9)
    ax.set_xlabel("$v_x$  [m/s]")
    ax.set_ylabel("$f$  [m$^{-6}$ s$^3$]")
    ax.set_title("SI units")
    ax.legend(fontsize=9)

    ax = axes[1]                                   # ---- normalized ----
    x = vx / vth
    ax.plot(x, exact*vth**3/n,   lw=2.2, color="C0", label="exact $f$")
    ax.plot(x, iso*vth**3/n,     lw=1.6, color="C1", ls="--", label="$f_0$")
    ax.plot(x, twoterm*vth**3/n, lw=1.6, color="C2", ls="-.", label="two-term")
    ax.plot(x, f1part*vth**3/n,  lw=1.2, color="C3", ls=":",  label="$f_1\cdot\hat{v}$")
    ax.axvline(V_frac, color="grey", lw=0.9)
    ax.set_xlabel("$v_x / v_{th}$")
    ax.set_ylabel("$f\\,v_{th}^3 / n$")
    ax.set_title("Normalized units")
    ax.legend(fontsize=9)

    fig.suptitle(f"{species},   T = {T_eV:g} eV,   n = {n:.2g} m$^{{-3}}$,   "
                 f"$V_x/v_{{th}}$ = {V_frac:.2f}")
    fig.tight_layout()
    plt.show()

display(widgets.interactive_output(plot_cut, controls))

<>:18: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:19: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:31: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:18: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:19: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:31: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
/var/folders/dm/fpq2vxz16_9577l0qt42tjqw0000gn/T/ipykernel_81906/3152588389.

Output()

### Where is $\mathbf f_1$ largest?

Item 1 of the problem asks you to show that the magnitude of the anisotropic part peaks at
$v=v_{th}=\sqrt{kT/m}$ (marked in the figure) and that there
$$|\mathbf f_1|_{\max}=n\sqrt{\frac{m}{ekT}}\left(\frac{m}{2\pi kT}\right)^{3/2}|\mathbf V|.$$


In [8]:
def plot_f1(species, T_eV, n, V_frac):
    m = SPECIES[species]
    kT = kT_from_T(T_eV)
    vth = v_thermal(m, kT)
    Vx = V_frac * vth

    v = np.linspace(0, 4*vth, 800)
    f1 = f1_radial_si(v, n, m, kT) * Vx           # |f1_vec|, drift along x

    # analytic peak at v = v_th
    v_pk = vth
    f1_max = n*np.sqrt(m/(np.e*kT)) * (m/(2*np.pi*kT))**1.5 * Vx

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

    ax = axes[0]                                   # ---- SI ----
    ax.plot(v, f1, lw=2.2, color="C3")
    ax.axvline(v_pk, color="k", lw=0.9, ls="--")
    ax.plot([v_pk], [f1_max], "ko", ms=7)
    ax.annotate("peak at $v=v_{th}$", xy=(v_pk, f1_max),
                xytext=(v_pk*1.5, f1_max*0.85),
                arrowprops=dict(arrowstyle="->"))
    ax.set_xlabel("$v$  [m/s]")
    ax.set_ylabel("$|\ f_1|$  [m$^{-6}$ s$^3$]")
    ax.set_title("SI units")

    ax = axes[1]                                   # ---- normalized ----
    ax.plot(v/vth, f1*vth**3/n, lw=2.2, color="C3")
    ax.axvline(1.0, color="k", lw=0.9, ls="--")
    ax.plot([1.0], [f1_max*vth**3/n], "ko", ms=7)
    ax.set_xlabel("$v/v_{th}$")
    ax.set_ylabel("$|\ f_1|\\,v_{th}^3/n$")
    ax.set_title("Normalized units")

    fig.suptitle("Anisotropic part $\ f_1$: peak at $v=v_{th}=\\sqrt{kT/m}$")
    fig.tight_layout()
    plt.show()

display(widgets.interactive_output(plot_f1, controls))

<>:24: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<>:32: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<>:35: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<>:24: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<>:32: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<>:35: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
/var/folders/dm/fpq2vxz16_9577l0qt42tjqw0000gn/T/ipykernel_81906/4157656572.

Output()

### Two-dimensional picture

Map of the distribution in the $(v_x,v_y)$ plane. The exact $f$ is a Gaussian of the *same
width* as $f_0$ but **shifted** along $v_x$ by $V_x$ (filled contours), whereas the isotropic
$f_0$ is centred on the origin (dashed contours).

Because the geometry is symmetric about the drift axis, a cut in the $(v_x,v_z)$ plane would
look exactly the same as this one.


In [9]:
def plot_map(species, T_eV, n, V_frac):
    m = SPECIES[species]
    kT = kT_from_T(T_eV)
    vth = v_thermal(m, kT)
    Vx = V_frac * vth

    # ---- SI grid (m/s) ----
    vv = np.linspace(-4*vth, 4*vth, 400)
    VX, VY = np.meshgrid(vv, vv)
    pref = n * (m/(2*np.pi*kT))**1.5
    f_exact = pref*np.exp(-m*((VX - Vx)**2 + VY**2)/(2*kT))
    f_iso   = pref*np.exp(-m*(VX**2 + VY**2)/(2*kT))

    # ---- normalized grid (v_th) ----
    g = np.linspace(-4, 4, 400)
    GX, GY = np.meshgrid(g, g)
    prefn = (2*np.pi)**-1.5
    f_exact_n = prefn*np.exp(-((GX - V_frac)**2 + GY**2)/2)
    f_iso_n   = prefn*np.exp(-(GX**2 + GY**2)/2)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax = axes[0]                                   # ---- SI ----
    lv = np.linspace(0, f_exact.max(), 18)
    ax.contourf(VX, VY, f_exact, levels=lv, cmap="viridis")
    ax.contour(VX, VY, f_iso, levels=lv[1::3], colors="w",
               linewidths=0.8, linestyles="--")
    ax.annotate("", xy=(Vx + 1.6*vth, 0), xytext=(Vx, 0),
                arrowprops=dict(arrowstyle="-|>", color="red", lw=2))
    ax.text(Vx + 1.75*vth, 0.25*vth, "$\ V$", color="red", fontsize=12)
    ax.set_xlabel("$v_x$ [m/s]")
    ax.set_ylabel("$v_y$ [m/s]")
    ax.set_title("SI units:  $f$ (filled)  vs  $f_0$ (dashed)")
    ax.set_aspect("equal")

    ax = axes[1]                                   # ---- normalized ----
    lvn = np.linspace(0, f_exact_n.max(), 18)
    ax.contourf(GX, GY, f_exact_n, levels=lvn, cmap="viridis")
    ax.contour(GX, GY, f_iso_n, levels=lvn[1::3], colors="w",
               linewidths=0.8, linestyles="--")
    ax.annotate("", xy=(V_frac + 1.6, 0), xytext=(V_frac, 0),
                arrowprops=dict(arrowstyle="-|>", color="red", lw=2))
    ax.text(V_frac + 1.75, 0.25, "$\ V$", color="red", fontsize=12)
    ax.set_xlabel("$v_x/v_{th}$")
    ax.set_ylabel("$v_y/v_{th}$")
    ax.set_title("Normalized units")
    ax.set_aspect("equal")

    fig.suptitle(f"{species},   T = {T_eV:g} eV,   $V_x/v_{{th}}$ = {V_frac:.2f}")
    fig.tight_layout()
    plt.show()

display(widgets.interactive_output(plot_map, controls))

<>:30: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<>:43: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<>:30: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<>:43: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
/var/folders/dm/fpq2vxz16_9577l0qt42tjqw0000gn/T/ipykernel_81906/3832617446.py:30: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
  ax.text(Vx + 1.75*vth, 0.25*vth, "$\ V$", color="red", fontsize=12)
/var/folders/dm/fpq2vxz16_9577l0qt42tjqw0000gn/T/ipykernel_81906/3832617446.py:43: S

Output()

---
## 2. The speed distribution $f_s(v)$

$f_s(v)=4\pi v^2 f_0(v)$ is the distribution of *speeds* (a scalar quantity). Its maximum
defines the most probable speed $v_{mp}=\sqrt{2kT/m}$. Also shown as reference are the
thermal speed $v_{th}=\sqrt{kT/m}$ (location of the $\mathbf f_1$ peak) and the mean speed
$\langle v\rangle=\sqrt{8kT/\pi m}$ from item 2 of the problem.


In [10]:
def plot_speed(species, T_eV, n, V_frac):
    m = SPECIES[species]
    kT = kT_from_T(T_eV)
    vth = v_thermal(m, kT)
    vmp = v_most_probable(m, kT)
    vmean = v_mean(m, kT)

    v = np.linspace(0, 4*vth, 800)
    fs = speed_dist_si(v, n, m, kT)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

    ax = axes[0]                                   # ---- SI ----
    ax.plot(v, fs, lw=2.2, color="C0", label="$f_s(v)$")
    ax.axvline(vth,   color="C1", ls="--", lw=1.3, label="$v_{th}$")
    ax.axvline(vmp,   color="C2", ls="--", lw=1.3, label="$v_{mp}$")
    ax.axvline(vmean, color="C3", ls="--", lw=1.3, label="$\\langle v\\rangle$")
    ax.set_xlabel("$v$  [m/s]")
    ax.set_ylabel("$f_s$  [m$^{-4}$ s]")
    ax.set_title("SI units")
    ax.legend(fontsize=9)

    ax = axes[1]                                   # ---- normalized ----
    ax.plot(v/vth, fs*vth/n, lw=2.2, color="C0")
    for x0, c in [(1.0, "C1"), (np.sqrt(2), "C2"), (np.sqrt(8/np.pi), "C3")]:
        ax.axvline(x0, color=c, ls="--", lw=1.3)
    ax.set_xlabel("$v/v_{th}$")
    ax.set_ylabel("$f_s\\,v_{th}/n$")
    ax.set_title("Normalized units")

    fig.suptitle(f"{species},   T = {T_eV:g} eV    "
                 f"($v_{{mp}}/v_{{th}}=\\sqrt{2}$, "
                 f"$\\langle v\\rangle/v_{{th}}=\\sqrt{{8/\\pi}}\\approx1.596$)")
    fig.tight_layout()
    plt.show()

display(widgets.interactive_output(plot_speed, controls))

Output()

Error in callback <function _draw_all_if_interactive at 0x118fb9380> (for post_execute), with arguments args (),kwargs {}:


ValueError: 
v_{mp}/v_{th}=\sqrt2
                   ^
ParseSyntaxException: Expected \sqrt{value}, found '2'  (at char 19), (line:1, col:20)

ValueError: 
v_{mp}/v_{th}=\sqrt2
                   ^
ParseSyntaxException: Expected \sqrt{value}, found '2'  (at char 19), (line:1, col:20)

<Figure size 1200x420 with 2 Axes>

---
## 3. The energy distribution $f_e(u)$

With the kinetic energy $u=\tfrac m2 v^2$,
$$f_e(u)=\frac{2n}{\sqrt\pi}\left(\frac{1}{kT}\right)^{3/2}u^{1/2}e^{-u/kT}.$$
Its maximum is the most probable energy $u=kT/2$ (dashed marker). The horizontal axis is in eV.


In [11]:
def plot_energy(species, T_eV, n, V_frac):
    m = SPECIES[species]
    kT = kT_from_T(T_eV)

    u_eV = np.linspace(0, 8*T_eV, 800)
    u_J = q_e * u_eV
    fe_eV = energy_dist_si(u_J, n, m, kT) * q_e          # [m^-3 eV^-1]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

    ax = axes[0]                                   # ---- SI / physical ----
    ax.plot(u_eV, fe_eV, lw=2.2, color="C0")
    ax.axvline(T_eV/2, color="C3", ls="--", lw=1.3, label="$u=kT/2$")
    ax.set_xlabel("$u$  [eV]")
    ax.set_ylabel("$f_e$  [m$^{-3}$ eV$^{-1}$]")
    ax.set_title("SI / physical units")
    ax.legend(fontsize=9)

    ax = axes[1]                                   # ---- normalized ----
    ax.plot(u_J/kT, energy_dist_si(u_J, n, m, kT)*kT/n, lw=2.2, color="C0")
    ax.axvline(0.5, color="C3", ls="--", lw=1.3)
    ax.set_xlabel("$u/kT$")
    ax.set_ylabel("$f_e\\,kT/n$")
    ax.set_title("Normalized units")

    fig.suptitle(f"{species},   T = {T_eV:g} eV    (most probable energy $u=kT/2$)")
    fig.tight_layout()
    plt.show()

display(widgets.interactive_output(plot_energy, controls))

Output()

---
## 4. The energy probability function $f_p(u)$

$$f_p(u)=u^{-1/2}f_e(u)=\frac{2n}{\sqrt\pi}\left(\frac{1}{kT}\right)^{3/2}e^{-u/kT}$$
carries **no** $u^{1/2}$ prefactor: it is a pure exponential, so $\ln f_p$ is a **straight
line** in $u$, with slope $-1/kT$. This is the content of item 5 of the problem.


In [12]:
def plot_fp(species, T_eV, n, V_frac):
    m = SPECIES[species]
    kT = kT_from_T(T_eV)

    u_eV = np.linspace(0.0, 8*T_eV, 800)
    u_J = q_e * u_eV
    fp_eV = energy_prob_si(u_J, n, m, kT) * q_e**1.5      # [m^-3 eV^-3/2]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

    ax = axes[0]                                   # ---- SI / physical, semilog ----
    ax.semilogy(u_eV, fp_eV, lw=2.2, color="C0")
    ax.set_xlabel("$u$  [eV]")
    ax.set_ylabel("$f_p$  [m$^{-3}$ eV$^{-3/2}$]")
    ax.set_title("SI / physical units (semilog)")

    ax = axes[1]                                   # ---- normalized ----
    x = u_J / kT
    lnfp = np.log(energy_prob_si(u_J, n, m, kT))
    base = np.log(energy_prob_si(np.array([1.0*kT]), n, m, kT)[0])
    ax.plot(x, lnfp, lw=2.2, color="C0", label="$\\ln f_p$")
    ax.plot(x, base - (x - 1.0), "k--", lw=1.0, label="slope $-1$")
    ax.set_xlabel("$u/kT$")
    ax.set_ylabel("$\\ln f_p$  (SI)")
    ax.set_title("Straight line: slope $-1$ in $u/kT$")
    ax.legend(fontsize=9)

    fig.suptitle(f"{species},   T = {T_eV:g} eV    "
                 f"(slope in $u$ is $-1/kT$)")
    fig.tight_layout()
    plt.show()

display(widgets.interactive_output(plot_fp, controls))

Output()

---
## 5. Questions

**Exercise 1 — characteristic speeds.**
Pick a species and a temperature with the sliders.

**(a)** Plot $f_s(v)$ and describe how its position and width change when you vary $T$ and the species.

**(b)** Compute the **mean speed** $\langle v\rangle$ and the **most probable speed** $v_{mp}$ in two ways:

*i)* numerically from $f_s$, i.e. $\langle v\rangle=\frac1n\int_0^\infty v\,f_s(v)\,dv$ and $v_{mp}$ as
the location of the maximum of $f_s$;

*ii)* analytically from $\dfrac{d f_s}{dv}=0$ and from the integral.

Check that the two agree, and mark $v_{th}$, $v_{mp}$ and $\langle v\rangle$ on the plot.

**Exercise 2 — species dependence.**
Compare an electron with the heavy ions (H$^+$, He$^+$, Ar$^+$, Kr$^+$, Xe$^+$) at the same $T$.
What happens to $v_{th}$, $v_{mp}$ and $\langle v\rangle$? Which species has the wider distribution
at fixed $T$?

**Exercise 3 — validity of the two-term expansion.**
Increase $V_x/v_{th}$ towards 1. When and why does the two-term approximation stop reproducing the
exact drifting Maxwellian? Relate your answer to the assumption $|\mathbf V|\ll\sqrt{kT/m}$.

**Exercise 4 — the log plot.**
In the semilog plot of $f_p$ versus $u$ you should see a straight line. What is its slope? How does
it change with $T$?

**Exercise 5 — relation between $f_p$ and $g_0$.**
Sometimes the energy distribution function is defined differently. For example, one may write the
distribution function directly as a function of the energy $u=\tfrac m2 v^2$,
$$f_0(\mathbf r,v)=g_0(\mathbf r,u(v)),$$
with the normalization
$$2\pi\left(\frac{2}{m}\right)^{3/2}\int g_0(\mathbf r,u)\,u^{1/2}\,du=n(\mathbf r).$$

**Prove that** the energy probability function $f_p$ and $g_0$ are related by
$$f_p(\mathbf r,u)=2\pi\left(\frac{2}{m}\right)^{3/2}g_0(\mathbf r,u),$$
and equivalently that
$$g_0(\mathbf r,u)=\frac{f_p(\mathbf r,u)}{2\pi\left(\frac{2}{m}\right)^{3/2}}
=\frac{f_e(\mathbf r,u)}{2\pi\left(\frac{2}{m}\right)^{3/2}u^{1/2}},$$
so that $g_0$ is **not** the same as $f_p$ (nor as $f_e$), but differs from them by the constant
factor $2\pi(2/m)^{3/2}$. Explain why this constant factor is what makes $\ln f_p$ linear in $u$
while $\ln f_e$ is not.
